# Global E-commerce Sales Analysis — SQL

**Objective:** Answer business questions using SQL queries and interpret the results.

**How to run**
1. Keep the dataset CSV in the same folder as this notebook.
2. Update `DATA_FILE` below to the exact CSV filename.
3. Run all cells from top to bottom.

This notebook uses **DuckDB SQL** for the analysis. Python is used only to load the CSV and execute/display SQL results; the analysis itself is written in SQL.

## 1. Business questions

This notebook investigates:
- Overall sales, profit and order volume
- Sales by region and customer segment
- Product and category profitability
- Monthly sales trends
- Discount, shipping cost and profit relationships
- Payment method performance
- Quantity sold by category
- Top customers and products

In [8]:
%pip -q install duckdb pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import duckdb
import pandas as pd
from IPython.display import display

# Change this to your actual dataset filename
DATA_FILE = r"C:\Users\purus\OneDrive\Desktop\Project\Data Analytics\01_Dataset\global_ecommerce_sales.csv"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW sales AS SELECT * FROM read_csv_auto('{DATA_FILE}', header=True)")

# Preview the dataset and confirm column names
display(con.execute("SELECT * FROM sales LIMIT 5").df())
display(con.execute("DESCRIBE sales").df())

,Order_ID,Order_Date,Customer_Name,Customer_Segment,Country,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount_Percent,Total_Sales,Shipping_Cost,Profit,Payment_Method
0,ORD-11121,2023-01-02,Karen Suzuki,Corporate,United States,North America,Technology,Wireless Bluetooth Headphones,3,99.43,0,298.29,9.31,124.92,Cash on Delivery
1,ORD-11244,2023-01-02,John Johansson,Corporate,Spain,Europe,Technology,Mechanical Gaming Keyboard,4,97.93,20,313.38,14.31,83.62,Cash on Delivery
2,ORD-10325,2023-01-03,Jessica Garcia,Consumer,Mexico,North America,Office Supplies,Binder Clips Assorted 48pc,2,10.74,0,21.48,8.12,3.69,Credit Card
3,ORD-10467,2023-01-03,Clara Taylor,Corporate,Italy,Europe,Technology,Webcam HD 1080p,2,61.86,15,105.16,10.79,26.32,PayPal
4,ORD-11454,2023-01-05,Felix Thomas,Consumer,Italy,Europe,Furniture,Standing Desk Converter,4,330.67,20,1058.14,11.09,253.44,Credit Card


,column_name,column_type,null,key,default,extra
0,Order_ID,VARCHAR,YES,None,None,None
1,Order_Date,DATE,YES,None,None,None
2,Customer_Name,VARCHAR,YES,None,None,None
3,Customer_Segment,VARCHAR,YES,None,None,None
4,Country,VARCHAR,YES,None,None,None
5,Region,VARCHAR,YES,None,None,None
6,Product_Category,VARCHAR,YES,None,None,None
7,Product_Name,VARCHAR,YES,None,None,None
8,Quantity,BIGINT,YES,None,None,None
9,Unit_Price,DOUBLE,YES,None,None,None


## 2. Data quality checks

**Why:** Before analysis, check missing values, duplicate rows and date range. This helps avoid misleading results.

In [11]:
display(con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE Order_ID IS NULL) AS missing_order_id,
    COUNT(*) FILTER (WHERE Order_Date IS NULL) AS missing_order_date,
    COUNT(*) FILTER (WHERE Customer_Name IS NULL) AS missing_customer,
    COUNT(*) FILTER (WHERE Product_Name IS NULL) AS missing_product,
    COUNT(*) FILTER (WHERE Total_Sales IS NULL) AS missing_sales,
    COUNT(*) FILTER (WHERE Profit IS NULL) AS missing_profit
FROM sales;
""").df())

,total_rows,missing_order_id,missing_order_date,missing_customer,missing_product,missing_sales,missing_profit
0,2000,0,0,0,0,0,0


In [12]:
display(con.execute("""
SELECT COUNT(*) - COUNT(DISTINCT
    (Order_ID, Order_Date, Customer_Name, Product_Name, Total_Sales, Profit)
) AS possible_duplicate_rows
FROM sales;
""").df())

,possible_duplicate_rows
0,0


In [13]:
display(con.execute("""
SELECT
    MIN(CAST(Order_Date AS DATE)) AS first_order_date,
    MAX(CAST(Order_Date AS DATE)) AS last_order_date,
    COUNT(*) AS total_records
FROM sales;
""").df())

,first_order_date,last_order_date,total_records
0,2023-01-02,2025-12-31,2000


## 3. Overall business performance

**Business question:** What are the total sales, total profit, number of orders and quantity sold?

**Why:** These KPIs give a quick overview of business performance.

In [14]:
display(con.execute("""SELECT
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    COUNT(DISTINCT Order_ID) AS total_orders,
    SUM(Quantity) AS total_quantity_sold,
    ROUND(AVG(Total_Sales), 2) AS average_sales_per_record
FROM sales;
""").df())

,total_sales,total_profit,total_orders,total_quantity_sold,average_sales_per_record
0,484559.34,158872.32,2000,7115.0,242.28


## 4. Sales by region

**Business question:** Which region generates the highest sales?

**Why:** Helps compare regional performance and identify strong and weak markets.

In [15]:
display(con.execute("""SELECT
    Region,
    COUNT(DISTINCT Order_ID) AS order_count,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit
FROM sales
GROUP BY Region
ORDER BY total_sales DESC;
""").df())

,Region,order_count,total_sales,total_profit
0,Europe,503,137006.20,45672.16
1,North America,578,133876.38,45250.09
2,Asia Pacific,520,121707.51,39116.61
3,South America,191,46051.13,14680.98
4,Middle East & Africa,208,45918.12,14152.48


## 5. Top 10 products by profit

**Business question:** Which products generate the highest total profit?

**Why:** Helps identify products that contribute most to profitability.

In [16]:
display(con.execute("""SELECT
    Product_Name,
    SUM(Quantity) AS quantity_sold,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit
FROM sales
GROUP BY Product_Name
ORDER BY total_profit DESC
LIMIT 10;
""").df())

,Product_Name,quantity_sold,total_sales,total_profit
0,Ergonomic Office Chair,167.0,45405.15,15104.58
1,Standing Desk Converter,215.0,46614.22,14694.35
2,Corner L-Shaped Desk,146.0,41070.48,13663.00
3,Mesh Back Task Chair,213.0,38179.77,12820.11
4,Wireless Bluetooth Headphones,212.0,28258.70,10611.08
5,Mechanical Gaming Keyboard,227.0,24072.42,8381.19
6,Filing Cabinet 3-Drawer,170.0,23108.01,7041.80
7,Portable External SSD 1TB,185.0,18065.30,6556.75
8,Business Casual Blazer,160.0,14327.72,6004.33
9,Noise Cancelling Earbuds,138.0,16119.60,5973.01


## 6. Profit by product category

**Business question:** Which product category is the most profitable?

**Why:** Helps compare category-level profitability and guide product decisions.

In [17]:
display(con.execute("""SELECT
    Product_Category,
    COUNT(*) AS record_count,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(AVG(Profit), 2) AS average_profit
FROM sales
GROUP BY Product_Category
ORDER BY total_profit DESC;
""").df())

,Product_Category,record_count,total_sales,total_profit,average_profit
0,Furniture,507,256274.68,81171.57,160.10
1,Technology,567,139518.22,48268.65,85.13
2,Clothing & Accessories,413,69375.63,26112.94,63.23
3,Office Supplies,513,19390.81,3319.16,6.47


## 7. Monthly sales trend

**Business question:** How do sales change month by month?

**Why:** Helps identify growth, declines and possible seasonal patterns.

In [18]:
display(con.execute("""SELECT
    DATE_TRUNC('month', CAST(Order_Date AS DATE)) AS order_month,
    COUNT(DISTINCT Order_ID) AS order_count,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit
FROM sales
GROUP BY order_month
ORDER BY order_month;
""").df())

,order_month,order_count,total_sales,total_profit
0,2023-01-01,47,11275.20,3573.01
1,2023-02-01,52,16943.67,5980.06
2,2023-03-01,51,16056.45,5718.77
3,2023-04-01,48,12185.00,3994.25
4,2023-05-01,40,8218.47,2591.86
5,2023-06-01,59,16191.61,5240.34
6,2023-07-01,59,15667.73,5195.97
7,2023-08-01,69,12971.07,4216.32
8,2023-09-01,58,11989.72,3577.00
9,2023-10-01,63,17794.60,5573.05


## 8. Sales by customer segment

**Business question:** Which customer segment contributes the most sales?

**Why:** Helps understand which customer group contributes most to revenue.

In [19]:
display(con.execute("""SELECT
    Customer_Segment,
    COUNT(DISTINCT Order_ID) AS order_count,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(AVG(Total_Sales), 2) AS average_sales
FROM sales
GROUP BY Customer_Segment
ORDER BY total_sales DESC;
""").df())

,Customer_Segment,order_count,total_sales,total_profit,average_sales
0,Consumer,1006,256287.74,87300.36,254.76
1,Corporate,623,146050.39,44463.44,234.43
2,Home Office,371,82221.21,27108.52,221.62


## 9. Discount vs profit

**Business question:** How does discount percentage relate to profit?

**Why:** Helps investigate whether higher discounts are associated with lower or higher profit. This analysis shows association, not proof of causation.

In [20]:
display(con.execute("""SELECT
    Discount_Percent,
    COUNT(*) AS record_count,
    ROUND(AVG(Profit), 2) AS average_profit,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(AVG(Total_Sales), 2) AS average_sales
FROM sales
GROUP BY Discount_Percent
ORDER BY Discount_Percent;
""").df())

,Discount_Percent,record_count,average_profit,total_profit,average_sales
0,0,503,100.81,50705.28,260.37
1,5,488,94.53,46129.15,267.32
2,10,443,74.34,32932.51,232.81
3,15,312,52.89,16503.10,195.75
4,20,169,55.06,9305.38,238.05
5,25,67,38.34,2568.58,198.09
6,30,18,40.46,728.32,301.57


## 10. Sales by payment method

**Business question:** Which payment method accounts for the highest sales?

**Why:** Helps understand customer payment preferences and sales contribution.

In [21]:
display(con.execute("""SELECT
    Payment_Method,
    COUNT(DISTINCT Order_ID) AS order_count,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit
FROM sales
GROUP BY Payment_Method
ORDER BY total_sales DESC;
""").df())

,Payment_Method,order_count,total_sales,total_profit
0,Credit Card,797,192717.89,62323.24
1,PayPal,609,143112.87,47288.80
2,Cash on Delivery,301,77129.54,25704.89
3,Bank Transfer,293,71599.04,23555.39


## 11. Quantity sold by product category

**Business question:** Which category sells the highest quantity?

**Why:** Compares sales volume across categories. Quantity sold and sales value are different measures.

In [22]:
display(con.execute("""SELECT
    Product_Category,
    SUM(Quantity) AS total_quantity_sold,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit
FROM sales
GROUP BY Product_Category
ORDER BY total_quantity_sold DESC;
""").df())

,Product_Category,total_quantity_sold,total_sales,total_profit
0,Technology,2017.0,139518.22,48268.65
1,Furniture,1806.0,256274.68,81171.57
2,Office Supplies,1741.0,19390.81,3319.16
3,Clothing & Accessories,1551.0,69375.63,26112.94


## 12. Shipping cost vs profit

**Business question:** How do shipping costs relate to profit?

**Why:** Helps explore whether orders with higher shipping costs tend to have different profit values.

In [23]:
display(con.execute("""SELECT
    CASE
        WHEN Shipping_Cost < 20 THEN 'Low Shipping Cost'
        WHEN Shipping_Cost < 50 THEN 'Medium Shipping Cost'
        ELSE 'High Shipping Cost'
    END AS shipping_cost_group,
    COUNT(*) AS record_count,
    ROUND(AVG(Shipping_Cost), 2) AS average_shipping_cost,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(AVG(Profit), 2) AS average_profit
FROM sales
GROUP BY shipping_cost_group
ORDER BY average_shipping_cost;
""").df())

,shipping_cost_group,record_count,average_shipping_cost,total_sales,total_profit,average_profit
0,Low Shipping Cost,1826,11.71,375937.40,122125.13,66.88
1,Medium Shipping Cost,174,25.36,108621.94,36747.19,211.19


## 13. Top customers by profit

**Business question:** Which customers contribute the highest total profit?

**Why:** Helps identify high-value customers and understand where customer-retention efforts may matter.

In [24]:
display(con.execute("""SELECT
    Customer_Name,
    COUNT(DISTINCT Order_ID) AS order_count,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit
FROM sales
GROUP BY Customer_Name
ORDER BY total_profit DESC
LIMIT 10;
""").df())

,Customer_Name,order_count,total_sales,total_profit
0,Priya Jackson,2,3836.46,1390.64
1,Hanna Jones,1,3813.98,1370.08
2,Li Laurent,1,3007.62,1189.79
3,Sarah Chen,2,3184.56,1153.99
4,Felix Thompson,1,2835.36,1115.05
5,Priya Oliveira,1,2901.11,1054.26
6,Marco Santos,1,2620.28,1017.49
7,Lucas Nilsson,2,2734.69,1016.19
8,Nadia Wang,3,2958.97,1001.81
9,Clara Müller,1,2538.50,992.84


## 14. Top customers by sales

**Business question:** Which customers generate the highest sales value?

**Why:** Sales contribution and profit contribution are not always the same. This query compares customers by revenue.

In [25]:
display(con.execute("""SELECT
    Customer_Name,
    COUNT(DISTINCT Order_ID) AS order_count,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit
FROM sales
GROUP BY Customer_Name
ORDER BY total_sales DESC
LIMIT 10;
""").df())

,Customer_Name,order_count,total_sales,total_profit
0,Priya Jackson,2,3836.46,1390.64
1,Hanna Jones,1,3813.98,1370.08
2,Sarah Chen,2,3184.56,1153.99
3,Li Laurent,1,3007.62,1189.79
4,Nadia Wang,3,2958.97,1001.81
5,Priya Oliveira,1,2901.11,1054.26
6,Felix Thompson,1,2835.36,1115.05
7,Linda Weber,1,2762.40,657.86
8,Karen Sato,2,2738.18,884.82
9,Lucas Nilsson,2,2734.69,1016.19


## 15. Payment method by customer segment

**Business question:** Which payment methods are used by each customer segment?

**Why:** Helps understand payment preferences across customer groups.

In [26]:
display(con.execute("""SELECT
    Customer_Segment,
    Payment_Method,
    COUNT(DISTINCT Order_ID) AS order_count,
    ROUND(SUM(Total_Sales), 2) AS total_sales
FROM sales
GROUP BY Customer_Segment, Payment_Method
ORDER BY Customer_Segment, total_sales DESC;
""").df())

,Customer_Segment,Payment_Method,order_count,total_sales
0,Consumer,Credit Card,384,95105.62
1,Consumer,PayPal,317,76395.33
2,Consumer,Cash on Delivery,154,45100.95
3,Consumer,Bank Transfer,151,39685.84
4,Corporate,Credit Card,260,64392.69
5,Corporate,PayPal,175,40421.89
6,Corporate,Bank Transfer,91,20933.15
7,Corporate,Cash on Delivery,97,20302.66
8,Home Office,Credit Card,153,33219.58
9,Home Office,PayPal,117,26295.65


## 16. Final business summary

**Why:** Collect the headline KPIs in one result so they can be used in a project summary or dashboard.

In [27]:
display(con.execute("""SELECT
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    COUNT(DISTINCT Order_ID) AS total_orders,
    SUM(Quantity) AS total_quantity_sold,
    (SELECT Region FROM sales GROUP BY Region ORDER BY SUM(Total_Sales) DESC LIMIT 1) AS highest_sales_region,
    (SELECT Product_Name FROM sales GROUP BY Product_Name ORDER BY SUM(Profit) DESC LIMIT 1) AS most_profitable_product,
    (SELECT Product_Category FROM sales GROUP BY Product_Category ORDER BY SUM(Profit) DESC LIMIT 1) AS most_profitable_category,
    (SELECT Customer_Segment FROM sales GROUP BY Customer_Segment ORDER BY SUM(Total_Sales) DESC LIMIT 1) AS highest_sales_segment,
    (SELECT Payment_Method FROM sales GROUP BY Payment_Method ORDER BY SUM(Total_Sales) DESC LIMIT 1) AS top_payment_method
FROM sales;
""").df())

,total_sales,total_profit,total_orders,total_quantity_sold,highest_sales_region,most_profitable_product,most_profitable_category,highest_sales_segment,top_payment_method
0,484559.34,158872.32,2000,7115.0,Europe,Ergonomic Office Chair,Furniture,Consumer,Credit Card


## Conclusion

The SQL queries confirm the main business findings:

- The highest-sales region is Europe (137.0K), followed by North America (133.9K).
- The highest-sales customer segment is Consumer (256.3K).
- The most profitable category is Furniture (81.2K), and the most profitable product is Ergonomic Office Chair (15.1K).
- Credit Card has the highest sales among payment methods (192.7K), while margins are similar across all methods (about 32-33%).
- Orders with higher discounts have lower profit, so heavy discounting should be reviewed.

These results were used to design the Power BI dashboard.